In [1]:
import pandas as pd
import numpy as np
import os
import glob as glob
import sys
import json

parent_dir = os.path.abspath(os.path.join(os.path.dirname(os.getcwd())))
sys.path.append(parent_dir)
import cpg_harmonizer
import s3_loader
import harmonize_checker

In [2]:
%load_ext autoreload
%autoreload 2

# Enter Identifiers, Metadata

In [ ]:
project_name = "cpg0010-caie-drugresponse"
source_list = ['broad-az']
output_parent_directory = "/Users/eweisbar/Desktop/cpgtest"

# typically, per-dataset metadata
# if not dataset-wide, delete from here and create conditional entry below
# if unknown, comment out
per_dataset_manual = {
    'Plate_Size':96,
    'CP_Version':'other',
    'DOI_to_Cite':'10.1158/1535-7163.MCT-09-1148',
    'Year_Imaged': 2007,
    'Cell_Line_Name':'MCF-7', # paper lists other lines, but our best guess is data is only this one
    'Cell_Line_Type':'breast cancer',
    'Cell_Line_Modification':'None', # paper says there was also a line with transfected dominant-negative truncated p53 mutant gene
    # our best guess is that this data is only the unmodified MCF-7 line
    'Microscope_Name':"Molecular Devices ImageXpress 5000A",
    'Microscope_Binning': 1,
    'Microscope_Modality':'Widefield',
    'Microscope_Objective_Magnification':20,
    'Microscope_Objective_NA':.45,
    #'Microscope_Pixel_Size': 0,
    'Image_Bit_Depth': 16,
    'Image_Size_X':1280,
    'Image_Size_Y':1024,
    #'Timepoint_Primary_Treatment':0, # methods states 24 and 48 hours in the experiment
    #'Timepoint_Secondary_Treatment':0,
    #'Timepoint_Acquisition':0,
    'Treatment_Category':'Compound',
}
# don't have any record of excitation/emission values for microscope

# Join CPG Metadata

In [4]:
if len(source_list) == 1:
    output_directory = os.path.join(output_parent_directory, project_name, source_list[0], "workspace", "harmonized_metadata")
else:
    output_directory = os.path.join(output_parent_directory, project_name, "all", "workspace", "harmonized_metadata")
if not os.path.exists(output_directory):
    os.makedirs(output_directory, exist_ok=True)

metadata_paths = []
load_paths = []
for src in source_list:
    metadata_paths.extend(s3_loader.parse_s3_folder(f"{project_name}/{src}/workspace/metadata/platemaps/"))
    load_paths.extend(s3_loader.parse_s3_folder(f"{project_name}/{src}/workspace/load_data_csv/"))

project = cpg_harmonizer.Project(output_directory, "../output_structure.json", project_name=project_name)

In [5]:
main_csvs = []
main_csv_batch = []
for path in load_paths:
    if path.endswith("load_data.csv"):
        main_csvs.append(s3_loader.read_s3_file(path, sep = ","))
        main_csv_batch.append(path.split("/")[-3])

In [6]:
platemap_txt = []
platemap_csvs = []
platemap_csv_batch = []
platemap_txt_batch = []
platemap_txt_name = []
external_tsv = []


for path in metadata_paths:
    if path.endswith(".csv"):
        platemap_csvs.append(s3_loader.read_s3_file(path, sep = ","))
        platemap_csv_batch.append(path.split("/")[-2])
    if path.endswith(".txt"):
        platemap_txt.append(s3_loader.read_s3_file(path, sep = "\t"))
        platemap_txt_name.append(path.split("/")[-1].split(".")[0])
        platemap_txt_batch.append(path.split("/")[-3])
    if path.endswith(".tsv"):
        external_tsv.append(s3_loader.read_s3_file(path, sep = "\t"))

In [19]:
complete_df = project.run_conversion(
    main_csvs = main_csvs, 
    main_csv_batch = main_csv_batch, 
    platemap_csvs = platemap_csvs,  
    platemap_csv_batch = platemap_csv_batch, 
    platemap_txt= platemap_txt, 
    platemap_txt_batch = platemap_txt_batch,
    platemap_txt_name = platemap_txt_name,
    external_tsv = external_tsv
)

No external metadata found
Skipping external metadata
No concentration columns found — skipping concentration merge.
Harmonizing values of Label


# Additional cleaning steps

In [20]:
with open('../inferable_relationships.json', "r") as f:
    inferable_metadata = json.load(f)

if per_dataset_manual['CP_Version'] != 'other':
    if not all([x in inferable_metadata["Label"].keys() for x in complete_df['Label'].unique()]):
        print(f'Labels need to be corrected to match any of {list(inferable_metadata["Label"].keys())}')
        print(f"Current labels are {complete_df['Label'].unique()}")
    # infer metadata from known relationships
    for column in inferable_metadata:
        for entry in inferable_metadata[column]:
            for inferred_column in inferable_metadata[column][entry]:
                if not inferred_column in complete_df.columns:
                    complete_df[inferred_column] = np.nan
                    complete_df[inferred_column] = complete_df[inferred_column].astype('str')
                complete_df.loc[complete_df[column]==entry, inferred_column] = inferable_metadata[column][entry][inferred_column]
else:
    print("Did not infer label metadata because CP_Version not inferrable")
    print(f"Labels that need to be manually declared are {complete_df['Label'].unique()}")

Did not infer label metadata because CP_Version not inferrable
Labels that need to be manually declared are ['Actin' 'Tubulin' 'DAPI']


In [21]:
# add per-experiment metadata manually annotated above
for col, val in per_dataset_manual.items():
    complete_df[col] = val

# add source information inferred from file path
for source in source_list:
    complete_df.loc[complete_df['File Path'].str.contains(f"/{source}/"),'Source'] = source

In [22]:
# manually declare relationships for non-canonical protocol
complete_df.loc[complete_df["Label"] == "Tubulin", "Label_Reagent"] = "mouse anti–β-tubulin IgG and Alexa Fluor 488 donkey anti-mouse IgG"
complete_df.loc[complete_df["Label"] == "Tubulin", "Label_Molecule"] = "β-tubulin"
complete_df.loc[complete_df["Label"] == "Tubulin", "Label_Structure"] = "β-tubulin"
complete_df.loc[complete_df["Label"] == "Tubulin", "Label_Mechanism"] = "Antibody"

complete_df.loc[complete_df["Label"] == "DAPI", "Label_Reagent"] = "DAPI"
complete_df.loc[complete_df["Label"] == "DAPI", "Label_Molecule"] = "DNA"
complete_df.loc[complete_df["Label"] == "DAPI", "Label_Structure"] = "Nucleus"
complete_df.loc[complete_df["Label"] == "DAPI", "Label_Mechanism"] = "Dye"

complete_df.loc[complete_df["Label"] == "Actin", "Label_Reagent"] = "Phalloidin"
complete_df.loc[complete_df["Label"] == "Actin", "Label_Molecule"] = "f-Actin"
complete_df.loc[complete_df["Label"] == "Actin", "Label_Structure"] = "f-Actin"
complete_df.loc[complete_df["Label"] == "Actin", "Label_Mechanism"] = "Dye"

In [23]:
# report on un-harmonized columns
# use reported information to manually update ontology OR dataframe OR input metadata
# this cell does NOT save the harmonization results, just check them
# this allows you to make any necessary corrections without accidentally overwriting
# returns a view of what the dataframe will look like after final harmonization
extra_cols = harmonize_checker.check_columns('../harmonized_ontology.json',complete_df, ret="extra_cols")

print("View of data that will be kept")
complete_df[[x for x in complete_df.columns if x not in extra_cols]]

Removing columns not in ontology: ['Metadata_Row', 'Replicate', 'Label', 'User_Name', 'Plate_Map', 'Metadata_Col']
Adding missing ontology columns: ['Microscope_Excitation_Peak', 'Treatment_Secondary_Treatment', 'Timepoint_Secondary_Treatment', 'Treatment_InChIKey', 'Treatment_Solvent', 'Microscope_Emission_Peak', 'Treatment_PubChem_CID', 'Treatment_SMILES', 'Treatment_Control_Class', 'Timepoint_Primary_Treatment', 'Treatment_Mechanism', 'Microscope_Emission_Width', 'Timepoint_Acquisition', 'Microscope_Excitation_Width', 'Microscope_Pixel_Size', 'Treatment_Broad_Sample', 'Image_Position_Z']
View of data that will be kept


,Plate,Well,Site,Batch,File Path,File Name,Treatment_Primary_Treatment,Treatment_Concentration,Plate_Size,CP_Version,...,Microscope_Objective_NA,Image_Bit_Depth,Image_Size_X,Image_Size_Y,Treatment_Category,Source,Label_Reagent,Label_Molecule,Label_Structure,Label_Mechanism
0,Week1_22123,B04,3,Week1,https://cellpainting-gallery.s3.us-east-1.amaz...,Week1_150607_B04_s3_w408BE006A-BF34-457E-81A9-...,cytochalasin B,10.0,96,other,...,0.45,16,1280,1024,Compound,broad-az,Phalloidin,f-Actin,f-Actin,Dye
1,Week1_22123,B04,4,Week1,https://cellpainting-gallery.s3.us-east-1.amaz...,Week1_150607_B04_s4_w49FF7E7B1-F049-4994-BCA2-...,cytochalasin B,10.0,96,other,...,0.45,16,1280,1024,Compound,broad-az,Phalloidin,f-Actin,f-Actin,Dye
2,Week1_22123,B04,1,Week1,https://cellpainting-gallery.s3.us-east-1.amaz...,Week1_150607_B04_s1_w494DCA5C4-3531-497D-A8B0-...,cytochalasin B,10.0,96,other,...,0.45,16,1280,1024,Compound,broad-az,Phalloidin,f-Actin,f-Actin,Dye
3,Week1_22123,B04,2,Week1,https://cellpainting-gallery.s3.us-east-1.amaz...,Week1_150607_B04_s2_w4342F300D-60F8-4256-A637-...,cytochalasin B,10.0,96,other,...,0.45,16,1280,1024,Compound,broad-az,Phalloidin,f-Actin,f-Actin,Dye
4,Week1_22123,B03,1,Week1,https://cellpainting-gallery.s3.us-east-1.amaz...,Week1_150607_B03_s1_w41094DA98-6064-4EA2-B4A7-...,cytochalasin B,30.0,96,other,...,0.45,16,1280,1024,Compound,broad-az,Phalloidin,f-Actin,f-Actin,Dye
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7579,Week9_39301,F11,4,Week9,https://cellpainting-gallery.s3.us-east-1.amaz...,Week9_090907_F11_s4_w19580FF4D-DC3D-4BD0-93FE-...,DMSO,0.0,96,other,...,0.45,16,1280,1024,Compound,broad-az,DAPI,DNA,Nucleus,Dye
7580,Week9_39301,G11,1,Week9,https://cellpainting-gallery.s3.us-east-1.amaz...,Week9_090907_G11_s1_w1EDE534D2-FCEE-4F92-A30B-...,DMSO,0.0,96,other,...,0.45,16,1280,1024,Compound,broad-az,DAPI,DNA,Nucleus,Dye
7581,Week9_39301,G11,2,Week9,https://cellpainting-gallery.s3.us-east-1.amaz...,Week9_090907_G11_s2_w10B010F39-3B4B-4DCB-8E34-...,DMSO,0.0,96,other,...,0.45,16,1280,1024,Compound,broad-az,DAPI,DNA,Nucleus,Dye
7582,Week9_39301,G11,3,Week9,https://cellpainting-gallery.s3.us-east-1.amaz...,Week9_090907_G11_s3_w10394282C-6D3D-4E0E-9FA3-...,DMSO,0.0,96,other,...,0.45,16,1280,1024,Compound,broad-az,DAPI,DNA,Nucleus,Dye


In [24]:
print("View of data that will be removed")
complete_df[extra_cols]

View of data that will be removed


,Metadata_Row,Replicate,Label,User_Name,Plate_Map,Metadata_Col
0,1,1,Actin,URL_OrigActin,Week1_22123,4
1,1,1,Actin,URL_OrigActin,Week1_22123,4
2,1,1,Actin,URL_OrigActin,Week1_22123,4
3,1,1,Actin,URL_OrigActin,Week1_22123,4
4,1,1,Actin,URL_OrigActin,Week1_22123,3
...,...,...,...,...,...,...
7579,5,3,DAPI,URL_OrigDAPI,Week9_39301,11
7580,6,3,DAPI,URL_OrigDAPI,Week9_39301,11
7581,6,3,DAPI,URL_OrigDAPI,Week9_39301,11
7582,6,3,DAPI,URL_OrigDAPI,Week9_39301,11


In [25]:
# After correcting any warnings above, run to save harmonization
complete_df = harmonize_checker.check_columns('../harmonized_ontology.json',complete_df)

Removing columns not in ontology: ['Metadata_Row', 'Replicate', 'Label', 'User_Name', 'Plate_Map', 'Metadata_Col']
Adding missing ontology columns: ['Microscope_Excitation_Peak', 'Treatment_Secondary_Treatment', 'Timepoint_Secondary_Treatment', 'Treatment_InChIKey', 'Treatment_Solvent', 'Microscope_Emission_Peak', 'Treatment_PubChem_CID', 'Treatment_SMILES', 'Treatment_Control_Class', 'Timepoint_Primary_Treatment', 'Treatment_Mechanism', 'Microscope_Emission_Width', 'Timepoint_Acquisition', 'Microscope_Excitation_Width', 'Microscope_Pixel_Size', 'Treatment_Broad_Sample', 'Image_Position_Z']


In [26]:
saved_path = os.path.join(output_directory,f"{project_name}_harmonized_metadata.parquet")
complete_df.to_parquet(saved_path)